# ACE-Step 1.5: music candidates (Google Colab)

Generates a few candidate scores from `storyboard.json`'s `music` block
(caption, bpm, instrumental), at the film's total duration.

**Before running:**
1. `Runtime -> Change runtime type -> T4 GPU` (free tier).
2. Build the same dataset zip described in the project's README (your local
   `storyboard.json` and, optionally but recommended, `outputs/timing.json`
   for an accurate duration) - you'll upload it directly below, no Google
   Drive or Kaggle Dataset needed. No Hugging Face token required for this one.

**Real finding, confirmed before writing this notebook:** ACE-Step 1.5 has no
scriptable command-line interface - its `cli.py` is wizard/interactive only.
This notebook drives its Python API directly instead (`AceStepHandler`,
`LLMHandler`, `generate_music`). If these exact names have moved by the time
you run this, check `ACE-Step-1.5/docs/en/INFERENCE.md` in the cloned repo
for the current API and adjust the script cell below.

Install is via `uv`, not pip, and the project wants Python 3.11-3.12.

In [ ]:
!nvidia-smi

In [ ]:
# --- Upload your dataset zip (storyboard.json, optionally timing.json) ---
import os
import zipfile

from google.colab import files

INPUT_DIR = "/content/input"
os.makedirs(INPUT_DIR, exist_ok=True)

uploaded = files.upload()
for name in uploaded:
    if name.endswith(".zip"):
        with zipfile.ZipFile(name) as z:
            z.extractall(INPUT_DIR)
        print(f"extracted {name} -> {INPUT_DIR}")

In [ ]:
# --- Configuration ---
OUTPUT_DIR = "/content/music"
NUM_CANDIDATES = 3

In [ ]:
# --- Install uv, then ACE-Step 1.5 ---
!curl -LsSf https://astral.sh/uv/install.sh | sh
import os
os.environ["PATH"] = f"{os.path.expanduser('~')}/.local/bin:" + os.environ["PATH"]

!git clone https://github.com/ACE-Step/ACE-Step-1.5.git
%cd ACE-Step-1.5
!uv python install 3.11
!uv sync

In [ ]:
# --- Write the generation script (runs inside uv's managed environment) ---
script = f'''
import json
import os
import time

INPUT_DIR = "{INPUT_DIR}"
OUTPUT_DIR = "{OUTPUT_DIR}"
NUM_CANDIDATES = {NUM_CANDIDATES}
os.makedirs(OUTPUT_DIR, exist_ok=True)

with open(f"{{INPUT_DIR}}/storyboard.json") as f:
    storyboard = json.load(f)
music_spec = storyboard["music"]

duration = 60
timing_path = f"{{INPUT_DIR}}/timing.json"
if os.path.exists(timing_path):
    with open(timing_path) as f:
        timing = json.load(f)
    duration = round(timing["total_effective_duration_s"] + storyboard["end_card"]["duration_s"])

from acestep.handler import AceStepHandler
from acestep.llm_inference import LLMHandler
from acestep.inference import GenerationParams, GenerationConfig, generate_music

dit_handler = AceStepHandler()
dit_handler.initialize_service(project_root=".", config_path="acestep-v15-turbo", device="cuda")

llm_handler = LLMHandler()
llm_handler.initialize(checkpoint_dir=".", lm_model_path="acestep-5Hz-lm-0.6B", backend="vllm", device="cuda")

params = GenerationParams(
    caption=music_spec["caption"],
    bpm=music_spec["bpm"],
    instrumental=music_spec["instrumental"],
    duration=duration,
    seed=music_spec.get("seed", -1),
)
config = GenerationConfig(batch_size=NUM_CANDIDATES, audio_format="flac")

start = time.time()
result = generate_music(dit_handler, llm_handler, params, config, save_dir=OUTPUT_DIR)
elapsed_s = time.time() - start
print("generate_music() returned:", result)

import glob
produced = sorted(glob.glob(f"{{OUTPUT_DIR}}/*.flac")) + sorted(glob.glob(f"{{OUTPUT_DIR}}/*.wav"))
gpu_log = {{}}
for i, src in enumerate(produced, start=1):
    gpu_log[f"cand{{i}}"] = {{"gpu_minutes": round(elapsed_s / 60 / max(len(produced), 1), 2), "source": os.path.basename(src)}}
json.dump(gpu_log, open(f"{{OUTPUT_DIR}}/gpu_log.json", "w"), indent=2)
print(f"{{len(produced)}} candidate(s) written to {{OUTPUT_DIR}} in {{elapsed_s/60:.1f}} GPU minutes total")
'''
with open("generate_music.py", "w") as f:
    f.write(script)

In [ ]:
!uv run python generate_music.py

In [ ]:
# --- Rename/convert candidates to cand1.wav, cand2.wav, ... (this project's
# music.py --import-dir expects that exact pattern) ---
import glob
import os
import shutil
import subprocess

!apt-get -y -qq install ffmpeg

sources = [
    p for p in sorted(glob.glob(f"{OUTPUT_DIR}/*.flac")) + sorted(glob.glob(f"{OUTPUT_DIR}/*.wav"))
    if not os.path.basename(p).startswith("cand")
]
for i, src in enumerate(sources, start=1):
    dst = f"{OUTPUT_DIR}/cand{i}.wav"
    if src.endswith(".wav"):
        shutil.copy(src, dst)
    else:
        subprocess.run(["ffmpeg", "-y", "-i", src, dst], check=True)
    print(f"{src} -> {dst}")

In [ ]:
# --- Zip for download ---
import shutil
from google.colab import files

shutil.make_archive("/content/ace_step_candidates", "zip", OUTPUT_DIR)
files.download("/content/ace_step_candidates.zip")